### **Arquitetura da CNN2D from scratch e dos resultados do treino obtidos**

In [ ]:
from tensorflow import keras
from keras import layers
import numpy as np

In [ ]:
def build_cnn(n_mels=40, frames=153, n_classes=3): # para corresponder aos inputs do joão

    # Entrada: uma "imagem" de 40 x 153 com 1 canal.
    # É o espectrograma: eixo vertical = frequências (n_mels),
    # eixo horizontal = tempo (frames), 1 canal = preto e branco.
    inp = keras.Input(shape=(frames, n_mels, 1))
    
    #Bloco 1: procura padrões simples (16 filtros)

    x = layers.Conv2D(16, 3, padding="same", activation="relu")(inp)
    x = layers.BatchNormalization(momentum=0.9)(x)   
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.25)(x)
    #Bloco 2: procura padrões mais complexos (32 filtros)

    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization(momentum=0.9)(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.25)(x)
    #Bloco 3: 64 filtros para identificar padrões mais abstratost

    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization(momentum=0.9)(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.25)(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(n_classes, activation="softmax")(x)

    return keras.Model(inp, out)

## **Estratégia de treino utilizada na obtenção dos resultados**

Foi desenhada para resolver os dois maiores desafios deste problema (a deteção de sopros cardíacos da competição CirCor DigiScope): o desbalanceamento severo das classes e o desfasamento entre a unidade de entrada do modelo (janela de áudio) e a unidade de diagnóstico clínico (paciente).

- Entrada do Modelo (Janelas): Os áudios longos dos pacientes foram segmentados no pré-processamento em janelas curtas de espectrograma log-mel (com dimensão [frames, n_mels, 1]). A CNN 2D aprende características locais (tempo-frequência) analisando cada janela isolada.  

- Agregação por Paciente (to_patient): Clinicamente, não interessa se uma janela teve ruído; interessa o diagnóstico do indivíduo. Por isso, o script agrupa todas as janelas que partilham o mesmo pid e calcula a média das probabilidades previstas para esse doente:

    $$\bar{P}(\text{classe} \mid \text{paciente}) = \frac{1}{K} \sum_{k=1}^K P(\text{classe} \mid \text{janela}_k)$$

- Compensação de Classes Desbalanceadas no Treino (class_weight): Na base de dados, a distribuição dos pacientes está fortemente inclinada para a ausência de sopro:   Absent: $\approx 73.8\%$   Present: $\approx 19.0\%$   Unknown: $\approx 7.2\%$   Se a rede treinasse sem ajuste, tenderia a prever sempre Absent para minimizar o erro global. Para contrariar isso, o script calcula pesos de classe inversamente proporcionais às contagens do conjunto de treino:
    $$\text{weight}_i = \frac{N_{\text{total}}}{3 \times N_i}$$

**Otimização Orientada à Métrica da Competição (Weighted Accuracy - WA)** : A métrica oficial da competição PhysioNet atribui recompensas desiguais aos acertos com base no vetor de pesos METRIC_W = [5.0, 3.0, 1.0]:  
 -  Present: peso $5.0$ |  Unknown: peso $3.0$ |  Absent: peso $1.0$


 Esta otimização é implementada pelas:

- Regra de Decisão do Doente: No código original, para decidir a classe final, as probabilidades médias eram ponderadas por este vetor (score = pr * METRIC_W) antes de tirar o argmax;

- Callback Customizado (PatientWA): Em cada final de época, a função on_epoch_end corre a validação completa, agrega por paciente e calcula o WA real da validação (val_wa), permitindo monitorizar a métrica que verdadeiramente importa e não apenas a acurácia por janela;

**Gestão Dinâmica da Convergência (Callbacks)**: O treino utiliza dois mecanismos automáticos para evitar overfitting e garantir que a rede converge de forma estável:   
- ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3): Se o erro de validação (val_loss) não diminuir durante 3 épocas consecutivas, a taxa de aprendizagem (learning rate) é reduzida para metade (passando de $0.001$ para $0.0005$, depois $0.00025$, etc.), permitindo que a descida do gradiente refine os pesos.   
- EarlyStopping(monitor="val_wa", mode="max", patience=8, restore_best_weights=True): Vigia diretamente o val_wa ao nível do paciente. Se o modelo passar 8 épocas sem conseguir superar o seu recorde de WA, o treino é interrompido e os pesos da melhor época são automaticamente restaurados para o ficheiro final.

Isto está definido no ficheiro train_cnn2D.py.

## **Função de loss utilizada - categorical crossentropy**

**O que é a Categorical Crossentropy?**
- mede a discrepância entre a distribuição de probabilidades prevista pela rede (através da camada final com ativação Softmax) e a classe real.

Para uma única amostra cujo rótulo verdadeiro seja a classe $c$, a fórmula calcula:$$\mathcal{L} = -\log(\hat{p}_c)$$onde $\hat{p}_c \in [0, 1]$ é a probabilidade que a rede previu para a classe correta:Se a rede tiver a certeza absoluta de que é a classe correta ($\hat{p}_c \approx 1$), a perda é quase zero ($-\log(1) = 0$).
- Se a rede errar e atribuir uma probabilidade baixa à classe correta ($\hat{p}_c \to 0$), o valor de $-\log(\hat{p}_c)$ dispara para infinito, aplicando uma penalização severa aos pesos da rede através do gradiente.

Isto significa que, durante a descida do gradiente (Backpropagation), quando o modelo erra numa janela com sopro (Present), a perda gerada é muito amplificada. 

O otimizador Adam é assim forçado a dar tanta importância a aprender padrões das classes raras como das maioritárias.

# Resultados

40 epochs

+**=== VALIDAÇÃO (nível de paciente) ===**
**A_murmur: 0.6778===**


                  |True Present | True Unknown | True Absent
    Pred Present  |          27 |          4   |       62
    Pred Unknown  |          0  |          6   |       13
    Pred Absent   |          0  |          0   |       30 
    
Interpretação:

--> a tabela não me parece nada bem, porque tem imensos falsos positivos: preveu que o murmur estava presente em todos os casos. 

Mas então pq é que a WA é 67,7%?? Porque estamos a tribuir mais peso quando o modelo acerta nas predicts para o present ! O peso é 5.

# como podemos melhorar isto??

Podemos variar os limiares óticos que maximizam a formula da WA. O que são os limiares ótimos?? são os valores que escolhemos manualmente para comberter as probabilidades continuas queo moedlo nos dá numa decisão de 
- por exemplo: neste momento estamos a fazer:
$$\text{score} = [\bar{p}_{\text{present}} \times 5.0,\; \bar{p}_{\text{unknown}} \times 3.0,\; \bar{p}_{\text{absent}} \times 1.0]$$

isso significa que basta p_presente ser maior ou igual a 0.17 para classificar o mumur como presente.
Neste moemnto estamos a usar o argmax como limiar ´toico, mas podemos fazer kinda uma hiperparametrização para encontrar qualk é o mellhor limiar ótico que nos dá os melhores resultados.


Suavizar os pesos no treino usando a raiz quadrada, mas continuando a multipicar na mesma as predições finais como a métrica de avaliação pede.

            counts = np.bincount(ytr, minlength=3).astype(float)
            smoothed = np.sqrt(counts.sum() / (3.0 * counts))
            # Normaliza para que a média dos pesos continue 1.0
            smoothed = smoothed / smoothed.mean()
            class_weight = {i: float(smoothed[i]) for i in range(3)}
            print("Novos class_weights suavizados:", class_weight)

Assim aprende a calibrar melhor as probabilidade reais de cada classe durante o treino e é menos agressivo e polarizado.

Podemos tentar tamvém fazer uma agregação diferente dos pacinetes na medida em que neste moemnto estamos a fazer a média simples de todas as janelas de um paciente, podemos correr o risco de que mesmo que um paciente tenha 3 janelas temporais com sopro nitido em 20 janelasno total, a média fica muito baixa, ou seja, há sopro na mesam, ams esta presença fica diluida pelo facto de não ser visivel nas outras 17 janelas !

Para combater isto pordemos usar a média de apenas as top k janelas mais ativadas ou seja, as que tenham mais probabilidade de ter o murmur presnete para evitar que haja esta diluição.